#End to End Template 

In [1]:
#imports
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device =torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
#Data Loading
transform = transforms.Compose([
    transforms.ToTensor()
])

trainset =torchvision.datasets.FashionMNIST(
    root ='./data',train =True,download=True,transform=transform
)
trainloader = torch.utils.data.DataLoader(
    trainset,batch_size=64,shuffle=True
)

100%|██████████| 26.4M/26.4M [00:16<00:00, 1.63MB/s]


Extracting ./data/FashionMNIST/raw/train-images-idx3-ubyte.gz to ./data/FashionMNIST/raw



100%|██████████| 29.5k/29.5k [00:00<00:00, 150kB/s]


Extracting ./data/FashionMNIST/raw/train-labels-idx1-ubyte.gz to ./data/FashionMNIST/raw



100%|██████████| 4.42M/4.42M [00:06<00:00, 653kB/s] 


Extracting ./data/FashionMNIST/raw/t10k-images-idx3-ubyte.gz to ./data/FashionMNIST/raw



100%|██████████| 5.15k/5.15k [00:00<00:00, 2.86MB/s]


Extracting ./data/FashionMNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/FashionMNIST/raw



In [9]:
testset =torchvision.datasets.FashionMNIST(
    root ='./data',train =False,download=True,transform=transform
)
testloader = torch.utils.data.DataLoader(
    testset,batch_size=64,shuffle=False
)

In [5]:
print(trainset[0][0].shape)

torch.Size([1, 28, 28])


In [3]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, 3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, stride=2, padding=1)

        self.relu = nn.ReLU()

        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [6]:
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)


In [ ]:
epochs =2
for epoch in range(epochs):
    running_loss=0.0
    for images,labels in trainloader:
        images,labels=images.to(device),labels.to(device)

        #forward pass
        outputs= model(images)

        #loss
        loss =criterion(outputs,labels)

        #backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss+=loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")

Epoch 1, Loss: 494.8796
Epoch 2, Loss: 328.9110


In [12]:
model.eval()
correct=0.0
totals=0.0
with torch.no_grad():
    for images,labels in testloader:
        images,labels=images.to(device),labels.to(device)
        outputs = model(images)

        _,predicted=torch.max(outputs,1)
        correct+=(predicted==labels).sum().item()

        totals+=labels.size(0)

accuracy=100*correct/totals
print("Accuracy:",accuracy)
         


Accuracy: 87.16
